```{contents}
```

## The Transformer Encoder Layer (Deep Explanation)

---

### What is the Encoder?

The **encoder** is the part of the Transformer responsible for **understanding** the input sequence.
It converts raw tokens (e.g., words or subwords) into **context-rich vector representations** that capture meaning and relationships between words.

> Think of the encoder as a “reader” that builds deep contextual understanding of the entire input sentence.

---

### Encoder Stack Overview

A Transformer encoder is composed of **N identical layers** (in the original paper, N = 6).
Each layer has two main sublayers:

1. **Multi-Head Self-Attention**
2. **Feed-Forward Network (FFN)**

Each sublayer has:

* A **residual connection** around it
* Followed by **Layer Normalization**

---

###  Encoder Layer Structure

```
Input Embeddings + Positional Encoding
        │
 ┌───────────────────────────┐
 │ Multi-Head Self-Attention │
 └───────────────────────────┘
        │
    Add + Norm
        │
 ┌───────────────────────────┐
 │ Feed Forward Network (FFN)│
 └───────────────────────────┘
        │
    Add + Norm
        ↓
   Output to next encoder layer
```

---

### Step-by-Step Workflow

Let’s denote the input to the encoder layer as:

$$
X \in \mathbb{R}^{n \times d_{model}}
$$

Where:

* $n$ = sequence length
* $d_{model} = 512$ (base model dimension)

---

#### **Step 1: Multi-Head Self-Attention**

Each token in the input **attends to all other tokens** (including itself).
This allows every word to gather contextual information from the entire sequence.

#### Computation:

1. **Linear projections**:
   $$
   Q = XW_Q, \quad K = XW_K, \quad V = XW_V
   $$

2. **Scaled dot-product attention**:
   $$
   \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
   $$

3. **Multi-head parallelization**:
   Split $Q,K,V$ into $h=8$ heads, apply attention on each, and concatenate:

   $$
   \text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O
   $$

Result:
$$
Z = \text{MultiHead}(X, X, X)
$$
(Z) is the new contextual representation.

---

#### **Step 2: Residual Connection + Layer Normalization**

To stabilize training and preserve original input information:
$$
X' = \text{LayerNorm}(X + Z)
$$

* **Residual connection** → improves gradient flow, prevents vanishing
* **Layer normalization** → normalizes feature scales, stabilizes activations

---

#### **Step 3: Feed-Forward Network (FFN)**

Each position’s representation is processed **independently** using a two-layer MLP:

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

Dimensions:

* $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$
* $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$
* $d_{ff} = 2048$

Adds **non-linearity** and **mixes features** within each token vector.

---

#### **Step 4: Another Residual + Layer Normalization**

The FFN output is added back to its input:
$$
Y = \text{LayerNorm}(X' + \text{FFN}(X'))
$$

Output (Y) becomes the input to the next encoder layer.

---

### Encoder Layer Equations Summary

| Sub-layer      | Formula                               |
| -------------- | ------------------------------------- |
| Self-Attention | $Z = \text{MultiHead}(X, X, X)$     |
| Add + Norm     | $X' = \text{LayerNorm}(X + Z)$      |
| Feed Forward   | $F = \max(0, X'W_1 + b_1)W_2 + b_2$ |
| Add + Norm     | $Y = \text{LayerNorm}(X' + F)$      |

---

### Intuition (How It “Understands” Text)

Let’s say your input sentence is:

> “The cat sat on the mat.”

At the **first layer**, each word attends to all others → learns dependencies like:

* “cat” ↔ “sat”
* “on” ↔ “mat”

At **deeper layers**, attention captures abstract relationships:

* Subject-object links
* Grammar roles
* Semantic meaning

After all layers, every token embedding contains **information about the entire sentence**.

---

### Encoder Outputs

The final encoder output is a matrix:
$$
Z = [z_1, z_2, ..., z_n]
$$

Each $z_i$ is a **contextualized vector** representing one token with full sentence-level awareness.

This $Z$ is passed to:

* The **decoder** (in encoder-decoder models, like translation)
* Or directly to **classification heads** (in encoder-only models, like BERT)

---

### Encoder Layer Visualization (Conceptual)

```
Word Embeddings + Positional Encoding
│
├──► Self-Attention: Each word attends to all others
│        ↓
│   "cat" ⇄ "sat" ⇄ "mat" ⇄ "on"
│
├──► Add & Norm
│
├──► Feed Forward: Nonlinear mixing of features
│
└──► Add & Norm → Output to next layer
```

---

### Why Encoder Works So Well

| Feature             | Benefit                                     |
| ------------------- | ------------------------------------------- |
| **Self-Attention**  | Captures global relationships efficiently   |
| **Residuals**       | Stable gradient flow                        |
| **LayerNorm**       | Smooth training                             |
| **Parallelization** | Processes all tokens simultaneously         |
| **Stacking Layers** | Builds hierarchical, abstract understanding |

---

### Encoder in Different Models

| Model                        | Encoder Usage                                         |
| ---------------------------- | ----------------------------------------------------- |
| **BERT**                     | Encoder-only (for sentence understanding)             |
| **T5 / BART**                | Encoder-decoder (for text generation + summarization) |
| **GPT**                      | No encoder (decoder-only)                             |
| **ViT (Vision Transformer)** | Encoder-only on image patches                         |

---

**In Short:**

> Each encoder layer in a Transformer reads all tokens at once, decides how much attention each should pay to others, refines their meanings through non-linear transformations, and passes the enriched context upward — layer by layer — until every token “understands” the entire input.

